# Variant A — 2K Control Run (same-scale baseline for Variant D)

**Purpose:** isolate the data-size confound when comparing Variant D (trained on the CoT-complete subset) against the headline Variant A run (trained on 100,000 rows). This notebook trains the **same Variant A architecture** (DeBERTa-v3-base + classification head, no rationale segment) on the **exact same train/dev rows that Variant D uses** — ~689 train / ~76 dev, drawn from the 765 rows that have completed CoT traces.

Rows are loaded via `data/preprocess_d.load_cot_subset` + `split_cot_subset` so the row IDs match D bit-for-bit.

**Hyperparameters identical to Variant A** (`microsoft/deberta-v3-base`, lr=2e-5, batch=8×accum=4 → effective 32, epochs=3, weight_decay=0.01, fp16, max_seq_length=256), with `warmup_steps=4` recomputed for the smaller training set (matches D: ~689 train rows ÷ 32 × 3 epochs = 63 total steps; 6% ≈ 4).

**Evaluation** uses the same 7 benchmarks as A/B/C/D (e-SNLI test, SNLI, MultiNLI matched/mismatched, ANLI R1/R2/R3) so the resulting `eval_results.json` is directly comparable to `results/variant_d_*/eval_results.json`.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers datasets accelerate torch

In [ ]:
# Cell 2: Mount Google Drive and set project path
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

PROJECT_ROOT = "/content/drive/MyDrive/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", os.getcwd())

In [ ]:
# Cell 3: Verify GPU
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:     ", torch.cuda.get_device_name(0))
    print("GPU memory:   ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# Cell 4: Load the same 1,800 train / 200 dev rows that Variant D uses
from training.config_d import VariantDConfig
from data.preprocess_d import load_cot_subset, split_cot_subset

# Reuse VariantDConfig only for path/seed consistency — NONE of the D-specific
# fields (sub_config, blend_ratio, alpha, MLM, rationale columns) affect this run.
load_cfg = VariantDConfig()

full_df = load_cot_subset(load_cfg)
train_df, dev_df = split_cot_subset(full_df, dev_frac=load_cfg.dev_frac, seed=load_cfg.split_seed)

print(f"Train rows: {len(train_df):,}  (must match D)")
print(f"Dev rows:   {len(dev_df):,}  (must match D)")
print("\nTrain label balance:")
print(train_df["gold_label"].value_counts().sort_index())

In [ ]:
# Cell 5: Tokenizer + datasets (premise + hypothesis only — Variant A architecture)
from transformers import AutoTokenizer
from data.preprocess import NLIDataset

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_SEQ_LENGTH = 256                 # Variant A's setting (no rationale → shorter is fine)
label2id = {"entailment": 0, "neutral": 1, "contradiction": 2}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

train_dataset = NLIDataset(
    premises=train_df["Sentence1"].tolist(),
    hypotheses=train_df["Sentence2"].tolist(),
    labels=[label2id[lbl] for lbl in train_df["gold_label"].tolist()],
    tokenizer=tokenizer,
    max_length=MAX_SEQ_LENGTH,
)
eval_dataset = NLIDataset(
    premises=dev_df["Sentence1"].tolist(),
    hypotheses=dev_df["Sentence2"].tolist(),
    labels=[label2id[lbl] for lbl in dev_df["gold_label"].tolist()],
    tokenizer=tokenizer,
    max_length=MAX_SEQ_LENGTH,
)

print(f"Train dataset: {len(train_dataset):,}")
print(f"Eval dataset:  {len(eval_dataset):,}")

In [ ]:
# Cell 6: Load model (vanilla classification head — Variant A architecture)
from transformers import AutoModelForSequenceClassification

id2label = {v: k for k, v in label2id.items()}
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label,
)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
model = model.float()

print(f"Model loaded. Parameters: {model.num_parameters():,}")

In [ ]:
# Cell 7: Train
import numpy as np
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "results", "variant_a_2k")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,        # effective batch = 32, matches A/B/C/D
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=4,                       # recomputed for ~689 train rows: 689//32*3=63 steps; 6%~4
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    dataloader_num_workers=0,
    logging_steps=25,
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Cell 8: Save final model to Drive
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}")

In [ ]:
# Cell 9: Evaluate on the same 7 benchmarks as A/B/C/D
import json
import pandas as pd
from data.preprocess import NLIDataset, load_snli_test, load_multinli, load_anli
from evaluation.evaluate import evaluate_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# e-SNLI test (premise + hypothesis only)
test_csv = os.path.join(PROJECT_ROOT, "Datasets", "E-SNLI", "esnli_test.csv")
test_df_raw = pd.read_csv(test_csv, usecols=["gold_label", "Sentence1", "Sentence2", "Explanation_1"])
test_df_raw = test_df_raw[test_df_raw["gold_label"].isin({"entailment", "neutral", "contradiction"})].dropna(subset=["Sentence1", "Sentence2"]).reset_index(drop=True)

test_dataset = NLIDataset(
    premises=test_df_raw["Sentence1"].tolist(),
    hypotheses=test_df_raw["Sentence2"].tolist(),
    labels=[label2id[lbl] for lbl in test_df_raw["gold_label"].tolist()],
    tokenizer=tokenizer,
    max_length=MAX_SEQ_LENGTH,
)

results = {"variant": "A_2k", "train_samples": len(train_dataset)}
results["esnli_test"] = evaluate_dataset(model, test_dataset, batch_size=32, device=device)
print(f"e-SNLI test:        {results['esnli_test']:.4f}")

snli = load_snli_test(tokenizer, MAX_SEQ_LENGTH)
results["snli_test"] = evaluate_dataset(model, snli, 32, device)
print(f"SNLI test:          {results['snli_test']:.4f}")

for split_name, split_key in [("multinli_matched", "validation_matched"), ("multinli_mismatched", "validation_mismatched")]:
    ds = load_multinli(tokenizer, split=split_key, max_length=MAX_SEQ_LENGTH)
    results[split_name] = evaluate_dataset(model, ds, 32, device)
    print(f"{split_name:<19} {results[split_name]:.4f}")

for r in ("r1", "r2", "r3"):
    ds = load_anli(tokenizer, round_tag=r, max_length=MAX_SEQ_LENGTH)
    results[f"anli_{r}"] = evaluate_dataset(model, ds, 32, device)
    print(f"ANLI {r}:            {results[f'anli_{r}']:.4f}")

results_path = os.path.join(OUTPUT_DIR, "eval_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to: {results_path}")